# COnfluence extraction pipeline

In [42]:
CONFLUENCE_BASE_URL="https://kaseya.atlassian.net/wiki/"
CONFLUENCE_EMAIL="sgil@itglue.com"
CONFLUENCE_API_TOKEN=""

(irb): warning: already initialized constant Object::CONFLUENCE_BASE_URL
(irb):1: warning: already initialized constant Object::CONFLUENCE_EMAIL
(irb):1: warning: previous definition of CONFLUENCE_EMAIL was here
(irb):2: warning: already initialized constant Object::CONFLUENCE_API_TOKEN
(irb):2: warning: previous definition of CONFLUENCE_API_TOKEN was here


""

In [3]:
#!/usr/bin/env ruby
# frozen_string_literal: true

require "json"
require "uri"
require "net/http"
require "optparse"

# https://kaseya.atlassian.net/wiki/spaces/FE

false

In [4]:
# Usage:
#   ENV:
#     CONFLUENCE_BASE_URL   e.g., https://your-domain.atlassian.net/wiki  (Cloud)
#                            or  https://confluence.your-company.com       (Server/DC)
#     CONFLUENCE_EMAIL      (Cloud only) your login email
#     CONFLUENCE_API_TOKEN  (Cloud) API token; (Server/DC) personal access token (PAT)
#
#   ruby confluence_collect.rb --space-key=ENG [--ancestor-id=123456] [--limit=100]
#
# Notes:
# - For Cloud: use email + API token (basic auth).
# - For Server/DC with PAT: put PAT in CONFLUENCE_API_TOKEN and leave CONFLUENCE_EMAIL empty.
# - Outputs JSONL to STDOUT.

BASE_URL = CONFLUENCE_BASE_URL

EMAIL     = CONFLUENCE_EMAIL
API_TOKEN = CONFLUENCE_API_TOKEN
SPACE_KEY = "FE"

"FE"

In [21]:
options = {
  space_key: SPACE_KEY,
  ancestor_id: nil,
  limit: 500
}

# OptionParser.new do |opts|
#   opts.banner = "Usage: ruby confluence_collect.rb --space-key=KEY [--ancestor-id=ID] [--limit=N]"

#   opts.on("--space-key=KEY", "Confluence space key (the 'project')") { |v| options[:space_key] = v }
#   opts.on("--ancestor-id=ID", Integer, "Optional ancestor page id to restrict tree") { |v| options[:ancestor_id] = v }
#   opts.on("--limit=N", Integer, "Page size per request (default 100, max 250 in Cloud)") { |v| options[:limit] = v }
# end.parse!

abort "Missing --space-key" unless options[:space_key]

# Build a CQL query for flexibility (works in Cloud and DC when CQL is enabled):
# - Restrict to pages in a space (type=page)
# - Optionally restrict to descendants of an ancestor
cql_parts = []
cql_parts << %Q(space="#{options[:space_key]}")
cql_parts << "type=page"
if options[:ancestor_id]
  # Restrict to descendants of a page
  cql_parts << "ancestor=#{options[:ancestor_id]}"
end
cql = cql_parts.join(" AND ")

"space=\"FE\" AND type=page"

In [32]:
# Prefer Cloud v2 search endpoint if available; fallback to v1 CQL if needed.
# We'll try v2 first; if it fails, we switch to v1.
def search_v2_available?
  true
end

def build_req(uri)
  req = Net::HTTP::Get.new(uri)
  if EMAIL && EMAIL.strip != "" # Cloud basic auth
    token = ["#{EMAIL}:#{API_TOKEN}"].pack("m0")
    req["Authorization"] = "Basic #{token}"
  elsif ENV["CONFLUENCE_API_TOKEN"] # Server/DC PAT via header
    req["Authorization"] = "Bearer #{ENV['CONFLUENCE_API_TOKEN']}"
  end
  req["Accept"] = "application/json"
  req
end

def http_for(uri)
  http = Net::HTTP.new(uri.host, uri.port)
  http.use_ssl = uri.scheme == "https"
  http.read_timeout = 120
  http.open_timeout = 30
  http
end

def page_url(base_url, id)
  # Cloud wiki pretty links work with /pages/{id} or /spaces/{key}/pages/{id}
  "#{base_url.gsub(%r{/$}, '')}/spaces/#{SPACE_KEY}/pages/#{id}"
end

def fetch_all_v2(base_url:, cql:, limit:)
  # Cloud v2 search: /wiki/api/v2/search?cql=...&limit=...&cursor=...
  # Returns results with content.id, content.title, space.key, etc.
  cursor = nil
  all = []
  loop do
    q = { "cql" => cql, "limit" => limit }
    q["cursor"] = cursor if cursor
    uri = URI("#{base_url.gsub(%r{/$}, '')}/api/v2/search")
    uri.query = URI.encode_www_form(q)

    res = http_for(uri).request(build_req(uri))
    unless res.is_a?(Net::HTTPSuccess)
      raise "v2 search failed: #{res.code} #{res.body}"
    end

    body = JSON.parse(res.body)
    results = body.fetch("results", [])
    results.each do |r|
      content = r["content"] || {}
      next unless content["type"] == "page"

      yield({
        "id" => content["id"],
        "title" => content["title"],
        "url" => page_url(base_url, content["id"]),
        "spaceKey" => (content.dig("space", "key") || r.dig("space", "key")),
        "lastUpdated" => content.dig("version", "when") || r.dig("version", "when"),
        "ancestors" => content["ancestors"] # may be nil; v2 sometimes omits
      })
    end

    cursor = body.dig("links", "next")
    break unless cursor && !results.empty?
  end
end

def fetch_all_v1(base_url:, cql:, limit:)
  # v1 CQL: /wiki/rest/api/search?cql=...&limit=...&start=...
  start = 0
  loop do
    uri = URI("#{base_url.gsub(%r{/$}, '')}/rest/api/search")
    params = { "cql" => cql, "limit" => limit, "start" => start, "expand" => "content.version,content.space" }
    uri.query = URI.encode_www_form(params)

    res = http_for(uri).request(build_req(uri))
    unless res.is_a?(Net::HTTPSuccess)
      raise "v1 search failed: #{res.code} #{res.body}"
    end

    body = JSON.parse(res.body)
    links = body.fetch("_links")
    total_size = body.fetch("totalSize")
    results = body.fetch("results", [])
      puts "=========================================="
      puts links
      puts "=============================" * 50 
    results.each do |r|
      content = r["content"] || {}
      next unless content["type"] == "page"

      yield({
        "id" => content["id"],
        "title" => content["title"],
        "url" => page_url(base_url, content["id"]),
        "spaceKey" => content.dig("space", "key"),
        "lastUpdated" => content.dig("version", "when"),
        "ancestors" => content["ancestors"] # often absent in v1 search
      })
    end

    size  = results.size
    total = body["totalSize"].to_i
    start += size
      puts "SIZE: #{size} TOTAL: #{total} START: #{start}"
    break if size.zero? || start >= total
  end
end

:fetch_all_v1

In [22]:
options[:limit]

500

In [34]:
rows = []
fetch_all_v1(base_url: BASE_URL, cql: cql, limit: options[:limit]) do |row|
    puts row.to_json
    rows << row
end

{"base"=>"https://kaseya.atlassian.net/wiki", "context"=>"/wiki", "next"=>"/rest/api/search?next=true&cursor=_f_MjAw_sa_WyJcdDMyMjQ3MDE0NCBjQ0BJYzktbTxbS2ozRVhcXC5Gaj8gY3AiXQ%3D%3D&expand=content.version%2Ccontent.space&limit=500&start=500&cql=space%3D%22FE%22+AND+type%3Dpage", "self"=>"https://kaseya.atlassian.net/wiki/rest/api/search?expand=content.version%2Ccontent.space&cql=space%3D%22FE%22+AND+type%3Dpage"}
{"id":"988217384","title":"Strange AWS resources","url":"https://kaseya.atlassian.net/wiki/spaces/FE/pages/988217384","spaceKey":"FE","lastUpdated":"2024-09-25T08:11:30.412Z","ancestors":null}
{"id":"985071691","title":"MrFixItService - Account restore","url":"https://kaseya.atlassian.net/wiki/spaces/FE/pages/985071691","spaceKey":"FE","lastUpdated":"2024-05-16T15:40:59.895Z","ancestors":null}
{"id":"972226568","title":"9/05/2024 - 10/05/2024 K1 SSO Disablement and Subsequent Security Risk Incident","url":"https://kaseya.atlassian.net/wiki/spaces/FE/pages/972226568","spaceKey":

In [19]:
# begin
#   # Try v2 first
#   fetch_all_v2(base_url: BASE_URL, cql: cql, limit: options[:limit]) do |row|
#     puts row.to_json
#   end
# rescue => e
#   warn "[info] v2 search not available or failed (#{e.message}). Falling back to v1 CQL…"
#   fetch_all_v1(base_url: BASE_URL, cql: cql, limit: options[:limit]) do |row|
#     puts row.to_json
#   end
# end

In [35]:
rows.count

800

In [37]:
rows[-1]

{"id"=>"322470144", "title"=>"Firefox Extension / Chrome", "url"=>"https://kaseya.atlassian.net/wiki/spaces/FE/pages/322470144", "spaceKey"=>"FE", "lastUpdated"=>"2023-03-09T15:27:21.282Z", "ancestors"=>nil}

In [41]:
OUTPUT = File.join(__dir__, "bedrock", "confluence_rows.jsonl")
FileUtils.mkdir_p(File.dirname(OUTPUT))

File.open(OUTPUT, "w") do |f|
  rows.each do |row|
    f.puts row.to_json
  end
end

puts "Saved #{rows.size} rows to #{OUTPUT} (JSONL format)"

(irb): warning: already initialized constant Object::OUTPUT
(irb):1: warning: previous definition of OUTPUT was here


Saved 800 rows to ./bedrock/confluence_rows.jsonl (JSONL format)
